# 05 — Avaliação Comparativa**TCC: Detecção de Smishing em Idosos com Modelos de Linguagem Natural**Compara as três trilhas sobre o **mesmo conjunto de teste** e produz osartefatos finais da seção 4.4.4. É o único notebook que enxerga os resultados detodos os modelos ao mesmo tempo.### Métricas| Métrica | Origem ||---|---|| Acurácia, Precisão, Recall, F1 | Declaradas na seção 4.3.4 || Matriz de confusão, **F2**, AUC-ROC | Acréscimos — precisam constar da 4.3.4 |**F2 é a métrica-título.** Um falso negativo deixa o golpe chegar ao idoso; umfalso positivo apenas sinaliza uma mensagem legítima. O F2 pondera o recall compeso 4x sobre a precisão, formalizando essa assimetria em um número único.Priorizar o recall isolado seria frágil: um classificador que responde "smishing"para tudo teria recall 1.0.

## 1. Setup

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
# No Colab CADA notebook roda em um runtime próprio: instalar dependências
# em um notebook não vale para os outros. Por isso esta célula se repete em
# todos, e não existe um "notebook de instalação".

REPO = 'https://github.com/FelypeSR/TCC_Cristian.git'   # ← ajuste aqui

!git clone -q {REPO} /content/TCC_Cristian 2>/dev/null || (cd /content/TCC_Cristian && git pull -q)
!pip install -q -r /content/TCC_Cristian/requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/TCC_Cristian/src')

import config as CFG
CFG.fixar_seeds()
CFG.criar_pastas()
CFG.resumo()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import evaluation as ev

teste = pd.read_csv(CFG.SPLIT_FILES['test'], encoding='utf-8')
predicoes = ev.carregar_predicoes()

## 2. Verificação de integridadeTodos os modelos precisam ter avaliado exatamente os mesmos exemplos. Qualquerdivergência aqui invalida a comparação — é o pressuposto que sustenta aafirmação de que as trilhas foram avaliadas em condições iguais.

In [ ]:
if not ev.verificar_integridade(predicoes):
    raise AssertionError(
        'Os modelos não avaliaram o mesmo conjunto. Reexecute os notebooks '
        'de modelo sobre o split atual antes de comparar.'
    )

## 3. Tabela comparativa

In [ ]:
tabela = ev.tabela_comparativa(predicoes)
exibir = ev.formatar_tabela(tabela)

print('=== Comparação sobre o conjunto de teste ===')
print(exibir.to_string())

# Destaca o melhor valor de cada métrica
display(exibir.style.highlight_max(axis=0, props='font-weight:bold; color:#c0392b'))

In [ ]:
print('=== Matriz de confusão por modelo ===\n')
for nome, df in predicoes.items():
    m = tabela.loc[nome]
    print(f'{nome}:  VP={int(m["VP"]):4}  FN={int(m["FN"]):4}  '
          f'FP={int(m["FP"]):4}  VN={int(m["VN"]):4}')
print('\nFN = golpe que passou (erro mais caro)  |  FP = legítima sinalizada')

## 4. Figuras

In [ ]:
ev.plot_confusao_comparativo(predicoes, tabela, salvar_como='05_confusao_comparativo.png')
plt.show()

In [ ]:
ev.plot_roc(predicoes, tabela, salvar_como='05_roc_comparativo.png')
plt.show()

In [ ]:
ev.plot_barras(tabela, salvar_como='05_barras_comparativo.png')
plt.show()

## 5. Significância estatísticaCom um conjunto de teste de 15% de um corpus pequeno, uma diferença de F1 de0.03 pode ser ruído. O teste de McNemar compara os **pares discordantes** — oscasos em que um modelo acerta e o outro erra.Comparar quatro modelos gera seis testes; sem correção, a chance de ao menos umfalso positivo passa de 25%. Por isso a correção de Holm.

In [ ]:
mcnemar = ev.mcnemar_todos(predicoes, alfa=0.05)

if len(mcnemar):
    print('=== Teste de McNemar (correção de Holm, α=0.05) ===')
    print(mcnemar[['modelo_a', 'modelo_b', 'so_a_acerta', 'so_b_acerta',
                   'p_valor', 'alfa_holm', 'significativo']].to_string(index=False))

    mcnemar.to_csv(f"{CFG.PATHS['metrics']}/mcnemar.csv", index=False, encoding='utf-8')

    n_sig = int(mcnemar['significativo'].sum())
    print(f'\nPares com diferença significativa: {n_sig} de {len(mcnemar)}')
    if n_sig == 0:
        print('Nenhuma diferença resistiu à correção — reporte isso no texto. '
              'É um resultado legítimo e mostra rigor.')

## 6. Análise qualitativa dos errosAtende à parte qualitativa declarada na seção 4.1 ("análise de característicaslinguísticas das mensagens fraudulentas") e ao que a seção 5 promete.

In [ ]:
analise = ev.analise_erros(predicoes, teste)
nomes = list(predicoes)

fn_unanimes = analise[(analise['rotulo_real'] == CFG.CLASSE_POSITIVA) & analise['erraram_todos']]
fp_unanimes = analise[(analise['rotulo_real'] == CFG.CLASSE_NEGATIVA) & analise['erraram_todos']]

print(f'Falsos negativos unânimes (golpe que escapou de TODOS): {len(fn_unanimes)}')
print(f'Falsos positivos unânimes (legítima sinalizada por TODOS): {len(fp_unanimes)}')

pd.set_option('display.max_colwidth', 130)

if len(fn_unanimes):
    print('\n=== Golpes que escaparam de todos os modelos ===')
    print('(o material mais rico da discussão — analise o que têm em comum)\n')
    colunas = ['id', 'texto'] + (['tipo_golpe'] if 'tipo_golpe' in analise.columns else [])
    display(fn_unanimes[colunas].head(10))

In [ ]:
# Taxa de erro por tipo de golpe — conecta com a etapa 4.4.1
por_tipo = ev.erros_por_tipo(analise, predicoes)
if len(por_tipo):
    print('=== Taxa de erro por tipo de golpe ===')
    print(por_tipo.to_string())

    fig, ax = plt.subplots(figsize=(11, 5))
    por_tipo.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('Taxa de erro')
    ax.set_xlabel('Tipo de golpe')
    ax.set_title('Taxa de erro por tipo de golpe e por modelo')
    ax.legend(title='Modelo')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f"{CFG.PATHS['figures']}/05_erros_por_tipo.png", dpi=150, bbox_inches='tight')
    plt.show()

    por_tipo.to_csv(f"{CFG.PATHS['metrics']}/erros_por_tipo.csv", encoding='utf-8')

In [ ]:
# Distribuição do número de modelos que erram cada mensagem
fig, ax = plt.subplots(figsize=(7, 4))
contagem = analise['n_erros'].value_counts().sort_index()
barras = ax.bar(contagem.index, contagem.values, color='#4878CF', edgecolor='white')
ax.set_xlabel('Número de modelos que erraram')
ax.set_ylabel('Número de mensagens')
ax.set_title('Distribuição de erros por mensagem')
ax.set_xticks(range(len(nomes) + 1))
for barra in barras:
    ax.text(barra.get_x() + barra.get_width() / 2, barra.get_height(),
            str(int(barra.get_height())), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CFG.PATHS['figures']}/05_distribuicao_erros.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Resumo final

In [ ]:
melhor_f2     = tabela['f2'].idxmax()
melhor_recall = tabela['recall'].idxmax()
melhor_auc    = tabela['auc_roc'].idxmax() if tabela['auc_roc'].notna().any() else '–'

print('=' * 62)
print('RESUMO — CONJUNTO DE TESTE')
print('=' * 62)
print(f'  Melhor F2 (métrica-título) : {melhor_f2}')
print(f'  Melhor Recall (smishing)   : {melhor_recall}')
print(f'  Melhor AUC-ROC             : {melhor_auc}')
print()
print(exibir.to_string())
print()
print('O F2 é a métrica-título: pondera o recall com peso 4x sobre a precisão,')
print('formalizando o custo assimétrico do erro. Um falso negativo deixa o golpe')
print('chegar ao idoso; um falso positivo apenas sinaliza uma mensagem legítima.')
print('=' * 62)

print('\n=== Arquivos gerados ===')
import os
for pasta in ['metrics', 'figures']:
    print(f'\n{pasta}/')
    for arquivo in sorted(os.listdir(CFG.PATHS[pasta])):
        print(f'  {arquivo}')